In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
# Install necessary packages
!pip install -q moviepy librosa ffmpeg-python

In [ ]:
# constants

# path where you have folders of the videos you need to segment
VIDEO_FOLDER_PATH = "path/to/Dataset"
SEGMENTS_EXTRACTION_PATH = "path/to/VideoSegments"

### Extract the Segments

In [4]:
import os
import moviepy.editor as mp
import librosa
import numpy as np
import ffmpeg
from tqdm.notebook import tqdm
from collections import defaultdict
from moviepy.video.io.VideoFileClip import VideoFileClip


# infomration list of extracted data file
meta_data = defaultdict(list)


def extract_folder_path(path: str) -> str:

    return path.split("/")[-1]


def extrat_video_name(video_name: str) -> str:

    return video_name.split(".")[0]


def extract_silent_timestamps(input_video_path: str, audio_output_path: str, top_db: int = 20 ):
    """
    Extracts non-silent intervals from the audio of a video file.

    This function processes a video to extract its audio track using the `moviepy` library, then analyzes the audio to find intervals of sound using the `librosa` library. Silent sections are determined based on a decibel threshold, and only non-silent timestamps are returned.

    :param input_video_path: str
        The file path to the input video from which audio will be extracted.

    :param audio_output_path: str
        The file path where the extracted audio will be saved temporarily.

    :param top_db: int, optional
        The threshold in decibels for considering sections as non-silent. Defaults to 20. A lower value means more sensitive detection, identifying quieter sounds as non-silent.

    :return: List[Tuple[float, float]]
        A list of tuples, each representing the start and end times (in seconds) of non-silent intervals detected in the audio.

    Notes:
    - This method extracts and temporarily saves the audio from the video file, which is subsequently analyzed. Ensure that there is enough storage space for this temporary file.
    - The function removes any transient silent intervals that fall under the specified top_db threshold, which helps in focusing on meaningful non-silent parts of the audio.
    - Requires `moviepy` and `librosa` to be installed in the Python environment.
    """
    # Step 1: Extract audio from video using moviepy
    clip = mp.VideoFileClip(input_video_path)
    audio = clip.audio
    audio.write_audiofile(audio_output_path)

    # Step 2: Load audio and detect non-silent intervals using librosa
    y, sr = librosa.load(audio_output_path, sr=None)
    non_silent_intervals = librosa.effects.split(y, top_db=top_db)

    # Convert non-silent intervals to seconds
    non_silent_intervals_sec = [(float(start / sr), float(end / sr)) for start, end in non_silent_intervals]

    return non_silent_intervals_sec


def split_video(input_file: str, output_prefix: str, output_path: str, intervals: list, metainfo: tuple, metadata: dict = meta_data) -> dict:
    """
    Splits a video into multiple segments based on specified time intervals.

    :param input_file: The path to the input video file.
    :param output_prefix: The prefix for the output files.
    :param intervals: A list of tuples, each containing (start_time, end_time) in seconds.
    :param metainfo: list of metadata items
    :param metadata: metadata dictionary
    """
    with VideoFileClip(input_file) as video:
        for index, (start, end) in enumerate(intervals):
            # Calculate the duration for the current segment
            duration = end - start

            # Subclip the video
            subclip = video.subclip(start, end)

            # Construct the output filename
            output_file = f"{output_prefix}_part{index+1}.mov"

            # Set the output path
            set_output_path = os.path.join(output_path, output_file)
            print(metadata, type(metadata))

            # set the metadata
            metadata["video_name"].append(metainfo[0])
            metadata["video_root_folder"].append(metainfo[1])
            metadata["audio_name"].append(metainfo[2])
            metadata["audio_root_folder"].append(metainfo[3])
            metadata["video_segment_name"].append(output_file)
            metadata["video_segment_path"].append(set_output_path)
            metadata["label"].append("<Label: Note Number>")

            # Write the subclip to the file
            subclip.write_videofile(set_output_path, codec='libx264')

  if event.key is 'enter':



In [ ]:
for root, dir, files in os.walk(VIDEO_FOLDER_PATH):
    for file_ in tqdm(files):
        # video root folder
        video_root_folder = os.path.join(root, file_)
        # extract the root directory where you have videos
        root_folder = extract_folder_path(root)
        # segment output path
        segment_folder = os.path.join(SEGMENTS_EXTRACTION_PATH, root_folder)
        # create path if not exists
        if not os.path.exists(segment_folder):
            os.mkdir(segment_folder)
            print("Folder created: {}".format(segment_folder))
        # extract video name
        video_name = extrat_video_name(file_)
        # create folders for each of the video names
        video_folder_path = os.path.join(segment_folder, video_name)
        # create the folder
        if not os.path.exists(video_folder_path):
            os.mkdir(video_folder_path)
            print("Folder Created: {}".format(video_folder_path))
        # save the audio and extract time feature
        audio_file_name = video_name + ".wav"
        # audio file path
        audio_file_path = os.path.join(video_folder_path, audio_file_name)
        # save the audio and extract the time stamp
        try:
          extracted_time_stamps = extract_silent_timestamps(video_root_folder, audio_file_path)
          # save the video segments
          split_video(
              video_root_folder,
              video_name,
              video_folder_path,
              extracted_time_stamps,
              (file_, video_root_folder, audio_file_name, audio_file_path),
          )
        except:
          print(f"{video_name} could not be processed")
          continue

### Create the MetaData Dataframe

In [6]:
import pandas as pd


# create the dataframe
metadata_df = pd.DataFrame.from_dict(meta_data)
metadata_df.head()

,video_name,video_root_folder,audio_name,audio_root_folder,video_segment_name,video_segment_path,label
0,IMG_0037.MOV,/content/drive/MyDrive/1:1_Chinee_Bernabe/Kuli...,IMG_0037.wav,/content/drive/MyDrive/1:1_Chinee_Bernabe/Kuli...,IMG_0037_part1.mov,/content/drive/MyDrive/1:1_Chinee_Bernabe/Kuli...,<Label: Note Number>
1,IMG_0037.MOV,/content/drive/MyDrive/1:1_Chinee_Bernabe/Kuli...,IMG_0037.wav,/content/drive/MyDrive/1:1_Chinee_Bernabe/Kuli...,IMG_0037_part2.mov,/content/drive/MyDrive/1:1_Chinee_Bernabe/Kuli...,<Label: Note Number>
2,IMG_0037.MOV,/content/drive/MyDrive/1:1_Chinee_Bernabe/Kuli...,IMG_0037.wav,/content/drive/MyDrive/1:1_Chinee_Bernabe/Kuli...,IMG_0037_part3.mov,/content/drive/MyDrive/1:1_Chinee_Bernabe/Kuli...,<Label: Note Number>
3,IMG_0037.MOV,/content/drive/MyDrive/1:1_Chinee_Bernabe/Kuli...,IMG_0037.wav,/content/drive/MyDrive/1:1_Chinee_Bernabe/Kuli...,IMG_0037_part4.mov,/content/drive/MyDrive/1:1_Chinee_Bernabe/Kuli...,<Label: Note Number>
4,IMG_0037.MOV,/content/drive/MyDrive/1:1_Chinee_Bernabe/Kuli...,IMG_0037.wav,/content/drive/MyDrive/1:1_Chinee_Bernabe/Kuli...,IMG_0037_part5.mov,/content/drive/MyDrive/1:1_Chinee_Bernabe/Kuli...,<Label: Note Number>


### Save the DataFrame

In [ ]:
save_path = "path/to/VideoSegments"
save_file = "Kulingtang_metadata.csv"

# save the dataframe
metadata_df.to_csv(os.path.join(save_path, save_file), index = False)